## DTSA 5510 - Peer Graded Assignement Week 4 - Part 2  

Limitation(s) of sklearn’s non-negative matrix factorization library. [20 pts]  

1. Load the movie ratings data (as in the HW3-recommender-system) and use matrix factorization technique(s) and predict the missing ratings from the test data. Measure the RMSE. You should use sklearn library. [10 pts]  

2. Discuss the results and why sklearn's non-negative matrix facorization library did not work well compared to simple baseline or similarity-based methods we’ve done in Module 3. Can you suggest a way(s) to fix it? [10 pts]  

In [271]:
# import libraries

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import time
from sklearn.model_selection import train_test_split
from scipy.sparse import coo_matrix, csr_matrix
from sklearn.decomposition import NMF

In [272]:
# load data

MV_users = pd.read_csv('data/users.csv')
MV_movies = pd.read_csv('data/movies.csv')
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

from collections import namedtuple
Data = namedtuple('Data', ['users','movies','train','test'])
data = Data(MV_users, MV_movies, train, test)

In [273]:
# define a Recommender System class

class RecSys():
    def __init__(self,data):
        self.data=data
        self.allusers = list(self.data.users['uID'])
        self.allmovies = list(self.data.movies['mID'])
        self.genres = list(self.data.movies.columns.drop(['mID', 'title', 'year']))
        self.mid2idx = dict(zip(self.data.movies.mID,list(range(len(self.data.movies)))))
        self.uid2idx = dict(zip(self.data.users.uID,list(range(len(self.data.users)))))
        self.Mr=self.rating_matrix()
        self.Mm=None 
        self.sim=np.zeros((len(self.allmovies),len(self.allmovies)))
        
    def rating_matrix(self):
        """
        Convert the rating matrix to numpy array of shape (#allusers,#allmovies)
        """
        ind_movie = [self.mid2idx[x] for x in self.data.train.mID] 
        ind_user = [self.uid2idx[x] for x in self.data.train.uID]
        rating_train = list(self.data.train.rating)
        
        return np.array(coo_matrix((rating_train, (ind_user, ind_movie)), shape=(len(self.allusers), len(self.allmovies))).toarray())
    
    def gen_NMF(self, X, rank=5):
        """
        takes a matrix (X) (m x n) and returns the constructed matrix (Cr) from matrices W (w x r) and H (r x n)
        """
        model = NMF(n_components=rank, init='random', random_state=0, max_iter=500)
        model.fit(X)
        W = model.transform(X)
        H = model.components_
        Cr = np.dot(W, H)
        return Cr

    def predict(self, X=None, nmf=False):
        """
        Predict ratings in the test data. Returns predicted rating in a numpy array of size (# of rows in testdata,)
        if X matrix is given, predict from that matrix, else use self.Mr
        if nmf True then use non-neg matrix factorization
        """
        if not np.any(X):
            X = self.Mr
        if nmf:
            X = self.gen_NMF(X)

        # Get predictions from the reconstructed matrix A = WH
        yp = np.array([])
        for i in range(len(self.data.test)):
            uID = self.data.test.iloc[i]['uID']
            mID = self.data.test.iloc[i]['mID']

            uidx = self.uid2idx[uID]
            midx = self.mid2idx[mID]

            rating = X[uidx, midx]
            yp = np.append(yp, rating)
        return yp
    
    def rmse(self,yp):
        yp[np.isnan(yp)]=3 #In case there is nan values in prediction, it will impute to 3.
        yt=np.array(self.data.test.rating)
        return np.sqrt(((yt-yp)**2).mean())

In [274]:
# instantiate a recommender class object

rec = RecSys(data)
print('Utility matrix shape:', rec.Mr.shape)

Utility matrix shape: (6040, 3883)


In [275]:
# make predictions using NMF matrix

yp = rec.predict(nmf=True)
rmse = rec.rmse(yp)
print('RMSE with NMF prediction:', rmse)

RMSE with NMF prediction: 2.9914200043174923


### Results:  

Prediction using non-negative matrix factorization technique showed much worse performance than the most simple baseline prediction model with all predictions set to 3 (NMF RMSE = 2.99; $Y_p=3$ RMSE = 1.2586).  The poor performance is not due to a failure of the NMF technique, but due to an inherent problem with the underlying data and in particular with its sparsity.  The vast majority of the user-item matrix has 0 values.  In turn, the correct relation between most of the user-item interactions is 0.  The NMF technique correctly predicts results which are close to the 0 value relation (or less than 1) but are not meaningful results, and in turn, not surprisingly leads to poor RMSE performance.

Examination of the sparsity and the values of the original utility matrix, and of the completion matrix constructed using NMF, demonstrates this limitation on extremely sparse data.  The originaly utility matrix has a large number of 0 value observations which results in substantial 0 value results in the NMF completion matrix.  Futhermore, the completion matrix also has an extemely large number of observations with values < 1 which are unrealistic and meaningless.

In [276]:
print('Utility matrix shape:', rec.Mr.shape)
print('Number of observations with value = 0:', f'{(rec.Mr == 0).sum():,}', 'as % of total:', round((rec.Mr == 0).sum() / (rec.Mr.shape[0] * rec.Mr.shape[1]), 2))

Utility matrix shape: (6040, 3883)
Number of observations with value = 0: 22,753,174 as % of total: 0.97


In [277]:
Xr = rec.gen_NMF(rec.Mr)

print('Completion matrix shape:', Xr.shape)
print('Number of observations with value = 0:', f'{(Xr == 0).sum():,}', 'as % of total:', round((Xr == 0).sum() / (Xr.shape[0] * Xr.shape[1]), 2))
print('Number of observations with value < 1:', f'{(Xr < 1).sum():,}', 'as % of total:', round((Xr < 1).sum() / (Xr.shape[0] * Xr.shape[1]), 2), '(includes val=0)')

Completion matrix shape: (6040, 3883)
Number of observations with value = 0: 3,150,801 as % of total: 0.13
Number of observations with value < 1: 22,938,069 as % of total: 0.98 (includes val=0)


### Discussion:  

The relative paucity of information in the Utility Matrix and poor performance with NMF technique can be considered a "cold start" problem."  This similar situation is encountered with the start of a "new community" or new recommender system with little information on user-item interactions.  

Possible mitigation techniques and their pros/cons have been suggested.  Such techniques include imputing values for ratings to make the matrix more dense, modeling directly on observed ratings only, weighting user-user or item-item interactions more, adding additional information sources, and hybrid (content-based / collaborative) methods.  

As an example and experiment, the following employs a method of imputation followed by NMF factorization to observe the effect on prediction performance.

In [278]:
# Impute all the missing values to 3 then perform NMF and test RMSE

Mr3  = rec.Mr + (rec.Mr == 0) * 3

In [279]:
# make predictions using NMF matrix

yp = rec.predict(X=Mr3, nmf=True)
rmse = rec.rmse(yp)
print('RMSE with NMF prediction:', rmse)

c:\Users\snowm\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\decomposition\_nmf.py:1665: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(


RMSE with NMF prediction: 1.129402513707887


Predictions based on NMF of the Utility Matrix after 0 values have been imputed to 3 showed improvement over the method with all predictions set to 3 (NMF RMSE = 1.1294; NMF $Y_p=3$ RMSE 1.2586).  

The following demonstrates a different method to impute the missing values (with improved RMSE) and the effect of using NMF on those imputations (with further improved RMSE).

In [287]:
#  helper function to predict 0 values to rating based on user average and movie average

def impute_ratings(X, w=0.5):
    """
    takes a utility matrix X (m x n) returns imputed ratings for all missing values = 0 using an average of mean user rating and mean moving rating
    can specify weight of user or movie rating such that 1 is equal weighting otherwise w < 1 is the proportion weight to users, eg w=0.75 user weight = 0.75 and movie weight = 0.25
    """ 

    import numpy.ma as ma

    # get a matrix of all 0 values imputed to movie (column) non-zero mean
    m_avg = rec.Mr
    m_avg = np.where(m_avg==0, np.nan, m_avg)
    m_avg = np.where(np.isnan(m_avg), ma.array(m_avg, mask=np.isnan(m_avg)).mean(axis=0), m_avg)
    # some movies (219) will have all 0 value ratings - impute to row average
    m_avg = np.where(m_avg == 0, np.nan, m_avg)
    m_avg = np.where(np.isnan(m_avg), np.nanmean(m_avg, axis=1)[:, np.newaxis], m_avg)

    # get a matrix of all 0 values imputed to user (row) non-zero mean
    u_avg = rec.Mr
    u_avg = np.where(u_avg==0, np.nan, u_avg)
    # np.where(np.isnan(u_avg), ma.array(u_avg, mask=np.isnan(u_avg)).mean(axis=1), u_avg)  # ValueError: operands could not be broadcast together with shapes (6040,3883) (6040,) (6040,3883) 
    u_avg = np.where(np.isnan(u_avg), ma.array(u_avg, mask=np.isnan(u_avg)).mean(axis=1)[:, np.newaxis], u_avg)

    # "average" the user and movie averages together.  w=0.5 means equal weight.  w is the proportion ascribed to average user rating ie u_avg*0.5 and m_avg*0.5
    w = 0.5
    avg = u_avg * w + m_avg * (1-w)

    return avg

In [292]:
avg_ratings = impute_ratings(rec.Mr)
yp = rec.predict(X=avg_ratings, nmf=False)
rmse = rec.rmse(yp)
print('RMSE without NMF prediction:', rmse)

RMSE without NMF prediction: 0.9563200645486082


In [293]:
yp = rec.predict(X=avg_ratings, nmf=True)
rmse = rec.rmse(yp)
print('RMSE with NMF prediction:', rmse)

c:\Users\snowm\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\decomposition\_nmf.py:1665: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(


RMSE with NMF prediction: 0.9405239051488842
